In [1]:
import pandas as pd
import numpy as np
import SALibrary.SportsAnalytics as sa
import SALibrary.SimpleRatingSystem as srs
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
df = pd.read_csv('./data/f1_data_processed_full_imputed.csv')
df.head(5)

,RaceDate,Year,RaceName,FullName,DriverId,TeamName,TeamId,GridPosition,Position_Race,Podium_Finish,TotalLength,MaxQualSpeed,Time,Speed,Finished
0,2018-11-25 13:10:00,2018,Abu Dhabi Grand Prix,Lewis Hamilton,hamilton,Mercedes,mercedes,1.0,1.0,1,290455,55.710277,5980.382,48.567968,1
1,2018-11-25 13:10:00,2018,Abu Dhabi Grand Prix,Sebastian Vettel,vettel,Ferrari,ferrari,3.0,2.0,1,290455,55.516426,5982.963,48.547016,1
2,2018-11-25 13:10:00,2018,Abu Dhabi Grand Prix,Max Verstappen,max_verstappen,Red Bull Racing,red_bull,6.0,3.0,1,290455,55.246943,5993.088,48.464998,1
3,2018-11-25 13:10:00,2018,Abu Dhabi Grand Prix,Daniel Ricciardo,ricciardo,Red Bull Racing,red_bull,5.0,4.0,0,290455,55.355814,5995.761,48.443392,1
4,2018-11-25 13:10:00,2018,Abu Dhabi Grand Prix,Valtteri Bottas,bottas,Mercedes,mercedes,2.0,5.0,0,290455,55.615232,6028.339,48.181597,1


In [4]:
df.isnull().sum()

RaceDate           0
Year               0
RaceName           0
FullName           0
DriverId           0
TeamName           0
TeamId             0
GridPosition       0
Position_Race      0
Podium_Finish      0
TotalLength        0
MaxQualSpeed       0
Time             430
Speed            430
Finished           0
dtype: int64

In [5]:
df['Average_Speed'] = df.groupby(['Year', 'RaceName'])['MaxQualSpeed'].transform('mean')
df['Speed_Delta'] = df['MaxQualSpeed'] - df['Average_Speed']
df.drop(columns=['Average_Speed'], inplace=True, errors='ignore')

### 1 year data prediction

In [6]:
model_df = df[df['Year'] == 2019]
# model_df.dropna(inplace=True)
model_df.shape

(420, 16)

In [7]:
target = model_df['MaxQualSpeed']
#target = model_df['Speed_Delta']
categorical_factors = model_df[['RaceName', 'DriverId', 'TeamId']]
# float_factors = model_df[['GridPosition', 'TotalLength']]

# Train SRS model using SRS package
model_coeffs, RMSE = srs.srs_train(target, categorical_factors)

print("\n*** Model Coefficients ***\n")
print(model_coeffs)
print("\nRMSE = {}".format(RMSE))


*** Model Coefficients ***

        type                      name      coeff
0   RaceName      Abu Dhabi Grand Prix  -6.919731
1   RaceName     Australian Grand Prix   1.784007
2   RaceName       Austrian Grand Prix   5.445743
3   RaceName     Azerbaijan Grand Prix  -4.119569
4   RaceName        Bahrain Grand Prix  -1.183142
5   RaceName        Belgian Grand Prix   6.195003
6   RaceName      Brazilian Grand Prix   1.421093
7   RaceName        British Grand Prix   5.364130
8   RaceName       Canadian Grand Prix  -1.366761
9   RaceName        Chinese Grand Prix  -2.832207
10  RaceName         French Grand Prix   2.057013
11  RaceName         German Grand Prix   1.211983
12  RaceName      Hungarian Grand Prix  -5.472256
13  RaceName        Italian Grand Prix   9.943816
14  RaceName       Japanese Grand Prix   3.247153
15  RaceName        Mexican Grand Prix  -5.030471
16  RaceName         Monaco Grand Prix -15.449181
17  RaceName        Russian Grand Prix   0.829028
18  RaceName      Sin

: 

In [32]:
# Aggregate the mean absolute coefficients for each categorical variable
coeff_impact = model_coeffs.groupby('type')['coeff'].apply(lambda x: x.abs().mean())

print("\n*** Mean Absolute Coefficients by Categorical Variable ***\n")
print(coeff_impact)


*** Mean Absolute Coefficients by Categorical Variable ***

type
DriverId    2.434250e-01
RaceName    2.498358e-15
TeamId      4.027010e-01
constant    2.293724e-13
Name: coeff, dtype: float64


In [33]:
# Train models excluding one categorical variable at a time
# Exclude RaceName
model_df_no_race = model_df.drop(columns=['RaceName'])
model_coeffs_no_race, RMSE_no_race = srs.srs_train(target, model_df_no_race[['DriverId', 'TeamId']])

# Exclude DriverId
model_df_no_driver = model_df.drop(columns=['DriverId'])
model_coeffs_no_driver, RMSE_no_driver = srs.srs_train(target, model_df_no_driver[['RaceName', 'TeamId']])

# Exclude TeamId
model_df_no_team = model_df.drop(columns=['TeamId'])
model_coeffs_no_team, RMSE_no_team = srs.srs_train(target, model_df_no_team[['RaceName', 'DriverId']])

print("\n*** RMSE Comparisons ***\n")
print(f"RMSE excluding RaceName: {RMSE_no_race:.4f}")
print(f"RMSE excluding DriverId: {RMSE_no_driver:.4f}")
print(f"RMSE excluding TeamId: {RMSE_no_team:.4f}")


*** RMSE Comparisons ***

RMSE excluding RaceName: 0.7592
RMSE excluding DriverId: 0.7817
RMSE excluding TeamId: 0.7592


In [27]:
pred_df = srs.srs_predict(model_coeffs, model_df)
pred_df

,RaceDate,Year,RaceName,FullName,DriverId,TeamName,TeamId,GridPosition,Position_Race,Podium_Finish,TotalLength,MaxQualSpeed,Time,Speed,Finished,Speed_Delta,prediction
0,2018-11-25 13:10:00,2018,Abu Dhabi Grand Prix,Lewis Hamilton,hamilton,Mercedes,mercedes,1.0,1.0,1,290455,55.710277,5980.382,48.567968,1,1.123249,55.827091
1,2018-11-25 13:10:00,2018,Abu Dhabi Grand Prix,Sebastian Vettel,vettel,Ferrari,ferrari,3.0,2.0,1,290455,55.516426,5982.963,48.547016,1,0.929397,55.849986
2,2018-11-25 13:10:00,2018,Abu Dhabi Grand Prix,Max Verstappen,max_verstappen,Red Bull Racing,red_bull,6.0,3.0,1,290455,55.246943,5993.088,48.464998,1,0.659914,55.241865
3,2018-11-25 13:10:00,2018,Abu Dhabi Grand Prix,Daniel Ricciardo,ricciardo,Red Bull Racing,red_bull,5.0,4.0,0,290455,55.355814,5995.761,48.443392,1,0.768786,55.215330
4,2018-11-25 13:10:00,2018,Abu Dhabi Grand Prix,Valtteri Bottas,bottas,Mercedes,mercedes,2.0,5.0,0,290455,55.615232,6028.339,48.181597,1,1.028204,55.761167
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
415,2018-10-21 18:10:00,2018,United States Grand Prix,Daniel Ricciardo,ricciardo,Red Bull Racing,red_bull,4.0,16.0,0,308728,58.966351,NaN,NaN,0,0.491809,59.102843
416,2018-10-21 18:10:00,2018,United States Grand Prix,Romain Grosjean,grosjean,Haas F1 Team,haas,8.0,17.0,0,308728,58.493369,NaN,NaN,0,0.018827,57.980869
417,2018-10-21 18:10:00,2018,United States Grand Prix,Fernando Alonso,alonso,McLaren,mclaren,13.0,18.0,0,308728,57.852541,NaN,NaN,0,-0.622001,58.107341
418,2018-10-21 18:10:00,2018,United States Grand Prix,Esteban Ocon,ocon,Racing Point,force_india,6.0,19.0,0,308728,58.558606,NaN,NaN,0,0.084064,58.427328


In [29]:
y = pred_df['MaxQualSpeed']
y_pred = pred_df['prediction']

print('R^2 Score:', r2_score(y, y_pred))
print('MAE:', mean_absolute_error(y, y_pred))
print('RMSE:', mean_squared_error(y, y_pred, squared=False))

R^2 Score: 0.9891879386179003
MAE: 0.4248762154553344
RMSE: 0.7592115998347556


In [30]:
# Function to perform a permutation test
def permutation_test_rmse(target, categorical_factors, original_rmse, n_permutations=1000):
    permuted_rmses = []
    
    for _ in range(n_permutations):
        # Shuffle the target variable and convert it back to pandas Series
        permuted_target = pd.Series(np.random.permutation(target), index=target.index)
        
        # Train SRS model on the shuffled target
        permuted_model_coeffs, permuted_rmse = srs.srs_train(permuted_target, categorical_factors)
        permuted_rmses.append(permuted_rmse)
    
    # Calculate the p-value: proportion of permuted RMSEs that are less than or equal to the original RMSE
    p_value = np.mean(np.array(permuted_rmses) <= original_rmse)
    
    return p_value, permuted_rmses

# Train SRS model using SRS package to get the original RMSE
model_coeffs, RMSE = srs.srs_train(target, categorical_factors)

# Perform the permutation test using the original RMSE
p_value, permuted_rmses = permutation_test_rmse(target, categorical_factors, RMSE)

print(f"Permutation Test p-value: {p_value:.4f}")

if p_value < 0.05:
    print("The model's RMSE is statistically significant (p < 0.05).")
else:
    print("The model's RMSE is not statistically significant (p >= 0.05).")

Permutation Test p-value: 0.0000
The model's RMSE is statistically significant (p < 0.05).
